In [2]:
# chaging the working path directory to the main folder
import os 
# move up one level to project root folder
os.chdir('..')
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\Jay Narsingani\OneDrive\Side Projects\mets-risk-score


In [19]:
# Cell 0: Session Restore
# Swithced to local environment from google collab
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Paths — relative to project root
data_proc = 'data/processed/'
fig_dir   = 'outputs/figures/'
model_dir = 'outputs/models/'

# Load target variable
y = pd.read_csv(os.path.join(data_proc, 'y_msss.csv')).squeeze()

# Load all 5 imputed datasets
imputed = []
for i in range(5):
    df = pd.read_csv(os.path.join(data_proc, f'x_imputed_{i+1}.csv'))
    imputed.append(df)

# Load survey weights
cohort  = pd.read_csv(os.path.join(data_proc, 'modeling_cohort.csv'),
                      low_memory=False)
weights = cohort['WTMECPRP'].values

print(f"Session restored")
print(f"   y shape         : {y.shape}")
print(f"   Imputed datasets: {len(imputed)} x {imputed[0].shape}")
print(f"   Survey weights  : {weights.shape}  mean={weights.mean():,.0f}")
print(f"   y mean          : {y.mean():.3f}")
print(f"   y std           : {y.std():.3f}")

Session restored
   y shape         : (1203,)
   Imputed datasets: 5 x (1203, 231)
   Survey weights  : (1203,)  mean=35,855
   y mean          : 0.010
   y std           : 1.104


In [20]:
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

In [21]:
# Cell 1: Define domain subsets for model comparision

# Train three seperate models to answer: 
# Model 1 (biochemistry-only): can secondary lab markers alone predict MSSS?
# Model 2 (lifestyle-only): can non-lab information predict MSSS?
# Model 3 (full multimodal): best possible prediction combining all domains

# Using imputed dataset 1 as reference for column names
X_ref = imputed[0]

# domain definition
biochem_cols = [c for c in X_ref.columns if c.startswith('LBX') or c.startswith('LBD')]

lifestyle_cols = [c for c in X_ref.columns
 if c.startswith('SL')    # sleep
                  or c.startswith('DPQ')   # mental health
                  or c.startswith('SMQ') or c.startswith('SMD')  # smoking
                  or c.startswith('ALQ')   # alcohol
                  or c.startswith('PAD') or c == 'total_MET_minutes'  # activity
                  or c.startswith('DR')    # dietary
                  or c in ['DBD100', 'DBQ095Z']]  # food behavior

demographic_cols = [c for c in X_ref.columns
                    if c.startswith('RIA') or c.startswith('DMD')
                    or c.startswith('IND') or c in [
                        'RIDAGEYR', 'RIDEXMON', 'RIDRETH1', 'RIDRETH3',
                        'AIALANGA', 'FIALANG', 'MIALANG', 'SIALANG'
                    ]]

body_cols = [c for c in X_ref.columns
             if c.startswith('BMX') or c.startswith('BMI')
             or c.startswith('BPX') or c.startswith('BPA')]

# Full predictor set
all_cols = X_ref.columns.tolist()

# Report
print(f"   Biochemistry (Model 1) : {len(biochem_cols)} columns")
print(f"   Lifestyle    (Model 2) : {len(lifestyle_cols)} columns")
print(f"   Demographics           : {len(demographic_cols)} columns")
print(f"   Body/BP                : {len(body_cols)} columns")
print(f"   Full multimodal        : {len(all_cols)} columns")

# Check for any overlap or missing columns
accounted = set(biochem_cols + lifestyle_cols + demographic_cols + body_cols)
unaccounted = [c for c in all_cols if c not in accounted]
print(f" Unaccounted columns: {len(unaccounted)}")
if unaccounted:
    for col in unaccounted:
        print(f"{col}")

   Biochemistry (Model 1) : 44 columns
   Lifestyle    (Model 2) : 165 columns
   Demographics           : 12 columns
   Body/BP                : 10 columns
   Full multimodal        : 231 columns
 Unaccounted columns: 0


In [22]:
# Cell 2: Model comparision setup

# Three predictor sets for comparision
model_sets = {
    'Biochemistry only' : biochem_cols,
    'Lifestyle only': lifestyle_cols,
    'Full Multimodal': all_cols,
}

# Cross-validation strategy
# Startified by MSSS quartile to ensure balance folds
kf = KFold(n_splits = 5, shuffle=True, random_state = 42)

# Storage for resutls across all 5 imputed datasets
results = {name: {'r2':[], 'rmse':[], 'mae':[]}
            for name in model_sets}

print(f"   Models          : {list(model_sets.keys())}")
print(f"   CV folds        : 5")
print(f"   Imputed datasets: 5")
print(f"   Total model fits: {len(model_sets) * 5 * 5}")
print(f"\n   Predictor counts:")
for name, cols in model_sets.items():
    print(f"   {name:<25} : {len(cols)} columns")

   Models          : ['Biochemistry only', 'Lifestyle only', 'Full Multimodal']
   CV folds        : 5
   Imputed datasets: 5
   Total model fits: 75

   Predictor counts:
   Biochemistry only         : 44 columns
   Lifestyle only            : 165 columns
   Full Multimodal           : 231 columns


In [ ]:
# Cell 3: ElasticNet Baseline

# Cell 3: ElasticNet with Proper Cross-Validation
# Fits on training folds, evaluates on held-out test folds
# This gives honest out-of-sample performance estimates

from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import time

kf = KFold(n_splits=5, shuffle=True, random_state=42 + dataset_idx)

enet_results = {name: {'r2': [], 'rmse': [], 'mae': []}
                for name in model_sets}

print("ElasticNet — 5-Fold Cross Validation\n")
print(f"{'Model':<25} {'Dataset':>8} {'R2':>8} {'RMSE':>8} {'MAE':>8}")
print('-' * 62)

for dataset_idx, X_imp in enumerate(imputed):
    for model_name, cols in model_sets.items():

        X_sub = X_imp[cols].values
        y_val = y.values

        fold_r2, fold_rmse, fold_mae = [], [], []

        for train_idx, test_idx in kf.split(X_sub):
            X_train, X_test = X_sub[train_idx], X_sub[test_idx]
            y_train, y_test = y_val[train_idx], y_val[test_idx]

            # Scale on training data only — apply to test
            scaler   = StandardScaler()
            X_train  = scaler.fit_transform(X_train)
            X_test   = scaler.transform(X_test)

            # Find best hyperparameters on training fold
            enet_cv = ElasticNetCV(
                l1_ratio     = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
                alphas       = np.logspace(-4, 1, 20),
                cv           = 3,
                max_iter     = 5000,
                random_state = 42,
                n_jobs       = -1
            )
            enet_cv.fit(X_train, y_train)

            # Evaluate on held-out test fold
            y_pred = enet_cv.predict(X_test)
            fold_r2.append(r2_score(y_test, y_pred))
            fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            fold_mae.append(mean_absolute_error(y_test, y_pred))

        # Average across folds for this dataset
        mean_r2   = np.mean(fold_r2)
        mean_rmse = np.mean(fold_rmse)
        mean_mae  = np.mean(fold_mae)

        enet_results[model_name]['r2'].append(mean_r2)
        enet_results[model_name]['rmse'].append(mean_rmse)
        enet_results[model_name]['mae'].append(mean_mae)

        print(f"{model_name:<25} {dataset_idx+1:>8} {mean_r2:>8.4f} "
              f"{mean_rmse:>8.4f} {mean_mae:>8.4f}")

# Summary across 5 imputed datasets
print(f"\n{'Model':<25} {'R2 mean':>10} {'R2 std':>8} "
      f"{'RMSE mean':>10} {'MAE mean':>9}")
print('-' * 65)

for model_name in model_sets:
    r2s   = enet_results[model_name]['r2']
    rmses = enet_results[model_name]['rmse']
    maes  = enet_results[model_name]['mae']
    print(f"{model_name:<25} {np.mean(r2s):>10.4f} {np.std(r2s):>8.4f} "
          f"{np.mean(rmses):>10.4f} {np.mean(maes):>9.4f}")

In [ ]:
!pip install xgboost

In [31]:
# Cell 4: XGBoost — Full Run (All 5 Datasets)

from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import time
import json
import os

param_grid = {
    'max_depth'        : [3, 5],
    'learning_rate'    : [0.05, 0.1],
    'n_estimators'     : [100, 200],
    'subsample'        : [0.8],
    'colsample_bytree' : [0.8],
}

xgb_results = {name: {'r2': [], 'rmse': [], 'mae': []}
               for name in model_sets}

print("XGBoost — 5-Fold CV (All 5 Datasets)\n")
print(f"{'Model':<25} {'Dataset':>8} {'R2':>8} {'RMSE':>8} {'MAE':>8}")
print('-' * 62)

for dataset_idx, X_imp in enumerate(imputed):
    for model_name, cols in model_sets.items():
        X_sub = X_imp[cols].values
        y_val = y.values

        kf = KFold(n_splits=5, shuffle=True,
                   random_state=42 + dataset_idx)

        fold_r2, fold_rmse, fold_mae = [], [], []

        for train_idx, test_idx in kf.split(X_sub):
            X_train, X_test = X_sub[train_idx], X_sub[test_idx]
            y_train, y_test = y_val[train_idx], y_val[test_idx]

            xgb = XGBRegressor(
                random_state = 42,
                n_jobs       = -1,
                verbosity    = 0
            )

            grid_search = GridSearchCV(
                xgb,
                param_grid,
                cv      = 3,
                scoring = 'r2',
                n_jobs  = -1,
                verbose = 0
            )
            grid_search.fit(X_train, y_train)

            y_pred = grid_search.best_estimator_.predict(X_test)
            fold_r2.append(r2_score(y_test, y_pred))
            fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            fold_mae.append(mean_absolute_error(y_test, y_pred))

        mean_r2   = np.mean(fold_r2)
        mean_rmse = np.mean(fold_rmse)
        mean_mae  = np.mean(fold_mae)

        xgb_results[model_name]['r2'].append(mean_r2)
        xgb_results[model_name]['rmse'].append(mean_rmse)
        xgb_results[model_name]['mae'].append(mean_mae)

        print(f"{model_name:<25} {dataset_idx+1:>8} {mean_r2:>8.4f} "
              f"{mean_rmse:>8.4f} {mean_mae:>8.4f}")

print(f"\n{'Model':<25} {'R2 mean':>10} {'R2 std':>8} "
      f"{'RMSE mean':>10} {'MAE mean':>9}")
print('-' * 65)

for model_name in model_sets:
    r2s   = xgb_results[model_name]['r2']
    rmses = xgb_results[model_name]['rmse']
    maes  = xgb_results[model_name]['mae']
    print(f"{model_name:<25} {np.mean(r2s):>10.4f} {np.std(r2s):>8.4f} "
          f"{np.mean(rmses):>10.4f} {np.mean(maes):>9.4f}")

# Save results
os.makedirs('outputs/models', exist_ok=True)
xgb_summary = {}
for model_name in model_sets:
    xgb_summary[model_name] = {
        'r2_mean'  : float(np.mean(xgb_results[model_name]['r2'])),
        'r2_std'   : float(np.std(xgb_results[model_name]['r2'])),
        'rmse_mean': float(np.mean(xgb_results[model_name]['rmse'])),
        'mae_mean' : float(np.mean(xgb_results[model_name]['mae'])),
    }

with open('outputs/models/xgboost_results.json', 'w') as f:
    json.dump(xgb_summary, f, indent=2)

print(f"\nXGBoost results saved")

XGBoost — 5-Fold CV (All 5 Datasets)

Model                      Dataset       R2     RMSE      MAE
--------------------------------------------------------------
Biochemistry only                1   0.5001   0.7752   0.5810
Lifestyle only                   1   0.1118   1.0357   0.7709
Full Multimodal                  1   0.6277   0.6711   0.4848
Biochemistry only                2   0.5075   0.7700   0.5808
Lifestyle only                   2   0.1188   1.0315   0.7640
Full Multimodal                  2   0.6143   0.6792   0.4874
Biochemistry only                3   0.4989   0.7777   0.5954
Lifestyle only                   3   0.1290   1.0260   0.7600
Full Multimodal                  3   0.6163   0.6815   0.4950
Biochemistry only                4   0.4844   0.7834   0.5898
Lifestyle only                   4   0.0992   1.0357   0.7679
Full Multimodal                  4   0.6092   0.6806   0.4867
Biochemistry only                5   0.4704   0.7915   0.5953
Lifestyle only                 

In [ ]:
!pip install lightgbm

In [34]:
# Cell 5: LightGBM — 5-Fold CV (All 5 Datasets)

import lightgbm as lgb
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import json
import os

param_grid = {
    'max_depth'        : [3, 5],
    'learning_rate'    : [0.05, 0.1],
    'n_estimators'     : [100, 200],
    'subsample'        : [0.8],
    'colsample_bytree' : [0.8],
}

lgb_results = {name: {'r2': [], 'rmse': [], 'mae': []}
               for name in model_sets}

print("LightGBM — 5-Fold CV (All 5 Datasets)\n")
print(f"{'Model':<25} {'Dataset':>8} {'R2':>8} {'RMSE':>8} {'MAE':>8}")
print('-' * 62)

for dataset_idx, X_imp in enumerate(imputed):
    for model_name, cols in model_sets.items():
        X_sub = X_imp[cols].values
        y_val = y.values

        kf = KFold(n_splits=5, shuffle=True,
                   random_state=42 + dataset_idx)

        fold_r2, fold_rmse, fold_mae = [], [], []

        for train_idx, test_idx in kf.split(X_sub):
            X_train, X_test = X_sub[train_idx], X_sub[test_idx]
            y_train, y_test = y_val[train_idx], y_val[test_idx]

            lgbm = lgb.LGBMRegressor(
                random_state = 42,
                n_jobs       = -1,
                verbose      = -1
            )

            grid_search = GridSearchCV(
                lgbm,
                param_grid,
                cv      = 3,
                scoring = 'r2',
                n_jobs  = -1,
                verbose = 0
            )
            grid_search.fit(X_train, y_train)

            y_pred = grid_search.best_estimator_.predict(X_test)
            fold_r2.append(r2_score(y_test, y_pred))
            fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            fold_mae.append(mean_absolute_error(y_test, y_pred))

        mean_r2   = np.mean(fold_r2)
        mean_rmse = np.mean(fold_rmse)
        mean_mae  = np.mean(fold_mae)

        lgb_results[model_name]['r2'].append(mean_r2)
        lgb_results[model_name]['rmse'].append(mean_rmse)
        lgb_results[model_name]['mae'].append(mean_mae)

        print(f"{model_name:<25} {dataset_idx+1:>8} {mean_r2:>8.4f} "
              f"{mean_rmse:>8.4f} {mean_mae:>8.4f}")

print(f"\n{'Model':<25} {'R2 mean':>10} {'R2 std':>8} "
      f"{'RMSE mean':>10} {'MAE mean':>9}")
print('-' * 65)

for model_name in model_sets:
    r2s   = lgb_results[model_name]['r2']
    rmses = lgb_results[model_name]['rmse']
    maes  = lgb_results[model_name]['mae']
    print(f"{model_name:<25} {np.mean(r2s):>10.4f} {np.std(r2s):>8.4f} "
          f"{np.mean(rmses):>10.4f} {np.mean(maes):>9.4f}")

# Save results
os.makedirs('outputs/models', exist_ok=True)
lgb_summary = {}
for model_name in model_sets:
    lgb_summary[model_name] = {
        'r2_mean'  : float(np.mean(lgb_results[model_name]['r2'])),
        'r2_std'   : float(np.std(lgb_results[model_name]['r2'])),
        'rmse_mean': float(np.mean(lgb_results[model_name]['rmse'])),
        'mae_mean' : float(np.mean(lgb_results[model_name]['mae'])),
    }

with open('outputs/models/lightgbm_results.json', 'w') as f:
    json.dump(lgb_summary, f, indent=2)

print(f"\nLightGBM results saved")

LightGBM — 5-Fold CV (All 5 Datasets)

Model                      Dataset       R2     RMSE      MAE
--------------------------------------------------------------
Biochemistry only                1   0.4959   0.7782   0.5854
Lifestyle only                   1   0.1247   1.0291   0.7662
Full Multimodal                  1   0.6081   0.6885   0.4953
Biochemistry only                2   0.5101   0.7699   0.5825
Lifestyle only                   2   0.1261   1.0277   0.7606
Full Multimodal                  2   0.6195   0.6767   0.4915
Biochemistry only                3   0.5013   0.7766   0.5891
Lifestyle only                   3   0.1361   1.0222   0.7579
Full Multimodal                  3   0.6209   0.6769   0.4955
Biochemistry only                4   0.4956   0.7774   0.5817
Lifestyle only                   4   0.1186   1.0252   0.7566
Full Multimodal                  4   0.6010   0.6912   0.4951
Biochemistry only                5   0.4700   0.7943   0.5997
Lifestyle only                

In [35]:
# Cell 6: Model Comparison Table

import pandas as pd
import json

# Load all saved results
with open('outputs/models/elasticnet_results.json') as f:
    enet = json.load(f)
with open('outputs/models/xgboost_results.json') as f:
    xgb = json.load(f)
with open('outputs/models/lightgbm_results.json') as f:
    lgbm = json.load(f)

# Build comparison table
rows = []
for model_name in model_sets.keys():
    rows.append({
        'Predictor Set'    : model_name,
        'Algorithm'        : 'ElasticNet',
        'R2 Mean'          : enet[model_name]['r2_mean'],
        'R2 Std'           : enet[model_name]['r2_std'],
        'RMSE Mean'        : enet[model_name]['rmse_mean'],
        'MAE Mean'         : enet[model_name]['mae_mean'],
    })
    rows.append({
        'Predictor Set'    : model_name,
        'Algorithm'        : 'XGBoost',
        'R2 Mean'          : xgb[model_name]['r2_mean'],
        'R2 Std'           : xgb[model_name]['r2_std'],
        'RMSE Mean'        : xgb[model_name]['rmse_mean'],
        'MAE Mean'         : xgb[model_name]['mae_mean'],
    })
    rows.append({
        'Predictor Set'    : model_name,
        'Algorithm'        : 'LightGBM',
        'R2 Mean'          : lgbm[model_name]['r2_mean'],
        'R2 Std'           : lgbm[model_name]['r2_std'],
        'RMSE Mean'        : lgbm[model_name]['rmse_mean'],
        'MAE Mean'         : lgbm[model_name]['mae_mean'],
    })

comparison_df = pd.DataFrame(rows)

# Format for display
comparison_df['R2 (mean ± std)'] = (
    comparison_df['R2 Mean'].map('{:.3f}'.format) + ' ± ' +
    comparison_df['R2 Std'].map('{:.3f}'.format)
)
comparison_df['RMSE'] = comparison_df['RMSE Mean'].map('{:.3f}'.format)
comparison_df['MAE']  = comparison_df['MAE Mean'].map('{:.3f}'.format)

display_df = comparison_df[[
    'Predictor Set', 'Algorithm', 'R2 (mean ± std)', 'RMSE', 'MAE'
]]

print("Model Comparison Table")
print("(5-fold CV averaged across 5 multiply imputed datasets)\n")
print(display_df.to_string(index=False))

# Save as CSV for paper
comparison_df.to_csv('outputs/models/model_comparison_table.csv', index=False)
print(f"\nSaved to outputs/models/model_comparison_table.csv")

# Identify best model overall
best_idx = comparison_df['R2 Mean'].idxmax()
best_row = comparison_df.loc[best_idx]
print(f"\nBest model overall:")
print(f"   {best_row['Predictor Set']} — {best_row['Algorithm']}")
print(f"   R2={best_row['R2 Mean']:.3f}  RMSE={best_row['RMSE Mean']:.3f}")

Model Comparison Table
(5-fold CV averaged across 5 multiply imputed datasets)

    Predictor Set  Algorithm R2 (mean ± std)  RMSE   MAE
Biochemistry only ElasticNet   0.636 ± 0.000 0.648 0.513
Biochemistry only    XGBoost   0.492 ± 0.013 0.780 0.588
Biochemistry only   LightGBM   0.495 ± 0.013 0.779 0.588
   Lifestyle only ElasticNet  -0.027 ± 0.000 1.099 0.803
   Lifestyle only    XGBoost   0.113 ± 0.010 1.032 0.766
   Lifestyle only   LightGBM   0.126 ± 0.006 1.025 0.760
  Full Multimodal ElasticNet   0.736 ± 0.000 0.553 0.434
  Full Multimodal    XGBoost   0.612 ± 0.011 0.681 0.491
  Full Multimodal   LightGBM   0.613 ± 0.007 0.683 0.494

Saved to outputs/models/model_comparison_table.csv

Best model overall:
   Full Multimodal — ElasticNet
   R2=0.736  RMSE=0.553


In [36]:
# Cell 7: Save Best Model for SHAP Analysis

from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
import pickle
import os

print("Training final ElasticNet on full dataset (all 5 imputed, averaged)...")

# Average across all 5 imputed datasets for final model
X_avg = sum([imputed[i][all_cols].values for i in range(5)]) / 5
y_val = y.values

# Scale
scaler      = StandardScaler()
X_scaled    = scaler.fit_transform(X_avg)

# Train final model
final_enet = ElasticNetCV(
    l1_ratio     = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
    alphas       = np.logspace(-4, 1, 20),
    cv           = 5,
    max_iter     = 5000,
    random_state = 42,
    n_jobs       = -1
)
final_enet.fit(X_scaled, y_val)

r2_train = final_enet.score(X_scaled, y_val)
print(f"Final model R2 (train): {r2_train:.4f}")
print(f"Best alpha            : {final_enet.alpha_:.6f}")
print(f"Best l1_ratio         : {final_enet.l1_ratio_:.2f}")
print(f"Non-zero features     : {(final_enet.coef_ != 0).sum()} of {len(all_cols)}")

# Save model and scaler
os.makedirs('outputs/models', exist_ok=True)
with open('outputs/models/final_elasticnet.pkl', 'wb') as f:
    pickle.dump(final_enet, f)
with open('outputs/models/final_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(f"\nSaved:")
print(f"   outputs/models/final_elasticnet.pkl")
print(f"   outputs/models/final_scaler.pkl")

Training final ElasticNet on full dataset (all 5 imputed, averaged)...
Final model R2 (train): 0.7899
Best alpha            : 0.006952
Best l1_ratio         : 1.00
Non-zero features     : 97 of 231

Saved:
   outputs/models/final_elasticnet.pkl
   outputs/models/final_scaler.pkl
